## Data Preprocessing

In [2]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, mean_poisson_deviance
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb

conn = sqlite3.connect(r'C:\Users\Owner\dev\algobetting\infra\data\db\algobetting.db')

df = pd.read_sql_query("""
                        SELECT DISTINCT
                            f.*,
                            ms.shots,
                            ms.opp_shots
                        FROM 
                            fbref_team_goals_features_005_365 f
                        JOIN
                            fbref_match_summary ms
                                ON ms.match_url = f.match_url
                                AND ms.team = f.team
                        WHERE 
                            f.match_date > '2021-11-01'
                       """, conn)

df

,match_url,match_date,season,division,team,opp_team,is_home,goals,opp_goals,xg,...,opp_weighted_defense_opp_shots_on_target,opp_weighted_defense_opp_xg,opp_weighted_defense_opp_npxg,opp_weighted_defense_opp_touches_att_pen_area,opp_weighted_defense_opp_touches_att_3rd,opp_weighted_defense_opp_touches,opp_weighted_defense_opp_pens_won,opp_weighted_defense_opp_corner_kicks,shots,opp_shots
0,https://fbref.com/en/matches/d8efb6cc/Leiceste...,2025-04-07 00:00:00,2024-2025,Premier-League,Leicester City,Newcastle Utd,1,0,3,0.5,...,4.355476,1.303009,1.251797,24.197442,155.426749,602.362132,0.065146,4.995137,7.0,16.0
1,https://fbref.com/en/matches/d8efb6cc/Leiceste...,2025-04-07 00:00:00,2024-2025,Premier-League,Newcastle Utd,Leicester City,0,3,0,3.3,...,5.494694,1.887059,1.825313,31.816265,195.700380,690.196090,0.073388,6.049093,16.0,7.0
2,https://fbref.com/en/matches/53e359bb/Manchest...,2025-04-06 00:00:00,2024-2025,Premier-League,Manchester Utd,Manchester City,1,0,0,0.9,...,3.624739,1.283831,1.218377,18.396063,101.868168,517.512371,0.081817,3.457143,13.0,9.0
3,https://fbref.com/en/matches/53e359bb/Manchest...,2025-04-06 00:00:00,2024-2025,Premier-League,Manchester City,Manchester Utd,0,0,0,0.5,...,4.125325,1.549812,1.440821,22.861867,150.093587,588.030979,0.103199,5.190624,9.0,13.0
4,https://fbref.com/en/matches/f671e515/Tottenha...,2025-04-06 00:00:00,2024-2025,Premier-League,Tottenham,Southampton,1,3,1,2.1,...,6.432989,2.281046,2.105396,32.140872,176.702024,626.645503,0.143379,5.438184,12.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12901,https://fbref.com/en/matches/3537bd55/Celta-Vi...,2025-05-10 00:00:00,2024-2025,La-Liga,Sevilla,Celta Vigo,0,2,3,2.3,...,3.562417,1.135448,1.026313,18.241637,127.491568,598.672673,0.100880,4.139149,18.0,8.0
12902,https://fbref.com/en/matches/ed775615/Valencia...,2025-05-10 00:00:00,2024-2025,La-Liga,Valencia,Getafe,1,3,0,2.7,...,3.140696,1.272427,1.108446,17.558451,115.609838,594.839955,0.175962,4.067525,8.0,10.0
12903,https://fbref.com/en/matches/ed775615/Valencia...,2025-05-10 00:00:00,2024-2025,La-Liga,Getafe,Valencia,0,0,3,0.5,...,3.899444,1.344925,1.193799,20.162915,148.126348,624.160701,0.152573,5.009976,10.0,8.0
12904,https://fbref.com/en/matches/ce67853a/Las-Palm...,2025-05-09 00:00:00,2024-2025,La-Liga,Las Palmas,Rayo Vallecano,1,0,1,0.7,...,3.975350,1.405973,1.224663,20.504361,128.134520,559.365555,0.236948,4.831992,11.0,16.0


## Feature Engineering

In [3]:
def add_features(X, df):
    # Original gap metrics code
    attack_features = [col for col in X.columns if col.startswith('weighted_attack_')]
    defense_features = [col for col in X.columns if col.startswith('opp_weighted_defense_')]
    
    feature_pairs = []
    for attack_feature in attack_features:
        metric = attack_feature.replace('weighted_attack_', '')
        defense_feature = f'opp_weighted_defense_opp_{metric}'
        
        if defense_feature in defense_features:
            feature_pairs.append((attack_feature, defense_feature, metric))
    
    for attack_feature, defense_feature, metric in feature_pairs:
        # Gap metrics
        gap_feature_name = f'{metric}_gap'
        X[gap_feature_name] = X[attack_feature] - X[defense_feature]
        
        # Ratio metrics
        ratio_feature_name = f'{metric}_ratio'
        X[ratio_feature_name] = X[attack_feature] / X[defense_feature].replace(0, 0.001)
        X[ratio_feature_name] = X[ratio_feature_name].replace([np.inf, -np.inf], np.nan)
        X[ratio_feature_name] = X[ratio_feature_name].fillna(X[ratio_feature_name].mean())
        
        # Interaction terms
        interaction_feature_name = f'{metric}_interaction'
        X[interaction_feature_name] = X[attack_feature] * X[defense_feature]
        
        # Normalized gap
        norm_gap_feature_name = f'{metric}_norm_gap'
        denominator = X[attack_feature] + X[defense_feature]
        X[norm_gap_feature_name] = X[gap_feature_name] / denominator.replace(0, 0.001)
        X[norm_gap_feature_name] = X[norm_gap_feature_name].replace([np.inf, -np.inf], np.nan)
        X[norm_gap_feature_name] = X[norm_gap_feature_name].fillna(X[norm_gap_feature_name].mean())

    
    return X

# Usage:
df = add_features(df.copy(), df)

## Model Training

In [4]:
X = df.drop(columns=["team", "opp_team", "goals", "opp_goals", "match_url",  "match_date", "xg", "opp_xg", "shots", "opp_shots"])
X = pd.get_dummies(X, columns=["season", "division"], drop_first=True)
y = pd.to_numeric(df["shots"], errors='coerce')

X_train = X[df['match_date'] < '2025-01-01']
X_test = X[df['match_date'] >= '2025-01-01']
y_train = y[df['match_date'] < '2025-01-01']
y_test = y[df['match_date'] >= '2025-01-01']

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set XGBoost parameters for Poisson regression
params = {
    'objective': 'count:poisson',  # Poisson regression for count data
    'eval_metric': 'poisson-nloglik',  # Poisson negative log-likelihood
    'seed': 26                     # Random seed
}

# Train the model
num_rounds = 100
model = xgb.train(
    params,
    dtrain,
    num_rounds,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=10,
    verbose_eval=50
)

# Make predictions
y_pred = model.predict(dtest)

# Evaluate model performance
# For Poisson, we can use Poisson deviance and RMSE
poisson_deviance = mean_poisson_deviance(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Poisson Deviance: {poisson_deviance:.4f}")
print(f"RMSE: {rmse:.4f}")

# Calculate mean absolute error for count data
mae = np.mean(np.abs(y_test - np.round(y_pred)))
print(f"MAE (rounded predictions): {mae:.4f}")

# Get feature importance
importance = model.get_score(importance_type='gain')
importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
print("\nFeature Importance:")
for feature, score in importance[:10]:
    print(f"{feature}: {score:.2f}")

[0]	train-poisson-nloglik:221.02368	test-poisson-nloglik:222.09993
[50]	train-poisson-nloglik:2.73987	test-poisson-nloglik:2.95000
[55]	train-poisson-nloglik:2.72033	test-poisson-nloglik:2.94963
Poisson Deviance: 1.6462
RMSE: 4.5221
MAE (rounded predictions): 3.5288

Feature Importance:
shots_interaction: 27.17
is_home: 24.07
touches_att_3rd_norm_gap: 7.39
goals_norm_gap: 6.29
pens_won_norm_gap: 5.48
shots_on_target_interaction: 5.39
npxg_gap: 5.27
npxg_norm_gap: 5.07
touches_att_3rd_interaction: 4.74
touches_att_pen_area_ratio: 4.57


## Parameter Tuning

In [5]:
import os
import pickle
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

# File to save best parameters
best_params_file = 'best_xgb_params.pkl'

# Check if we already have saved parameters
if os.path.exists(best_params_file):
    # Load previously found best parameters
    with open(best_params_file, 'rb') as f:
        best_params = pickle.load(f)
    print("Loaded saved parameters:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")
        
    # Create model with the best parameters
    best_model = xgb.XGBRegressor(
        objective='count:poisson',
        eval_metric='poisson-nloglik',
        random_state=26,
        n_jobs=-1,
        **best_params  # Unpack the best parameters
    )
    
    # Train the model on all training data
    best_model.fit(X_train, y_train)
    
else:
    # Run the parameter search if no saved parameters exist
    tuned_model = xgb.XGBRegressor(
        objective='count:poisson',
        eval_metric='poisson-nloglik',
        random_state=26,
        n_jobs=-1
    )
    
    # Parameter grid to search
    param_grid = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [3, 4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.2, 0.3, 0.4],
        'reg_alpha': [0, 0.1, 0.5, 1, 5],
        'reg_lambda': [0.1, 0.5, 1, 5, 10]
    }
    
    # Setup RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=tuned_model,
        param_distributions=param_grid,
        n_iter=500,
        scoring='neg_mean_poisson_deviance',
        cv=[(np.where(df['match_date'] < '2025-01-01')[0], 
             np.where(df['match_date'] >= '2025-01-01')[0])],
        verbose=1,
        random_state=26,
        n_jobs=-1
    )
    
    # Run the random search
    print("Running parameter search (this may take a while)...")
    random_search.fit(X, y)
    
    # Get the best parameters and model
    best_params = random_search.best_params_
    best_score = random_search.best_score_
    best_model = random_search.best_estimator_
    
    print(f"Best score: {best_score}")
    print("Best parameters:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")
    
    # Save the best parameters for future use
    with open(best_params_file, 'wb') as f:
        pickle.dump(best_params, f)
    print(f"Saved parameters to {best_params_file}")

# Continue with model evaluation regardless of how we got the model
y_pred = best_model.predict(X_test)

# Evaluate final model performance
poisson_deviance = mean_poisson_deviance(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = np.mean(np.abs(y_test - np.round(y_pred)))

print(f"\nFinal Model Performance:")
print(f"Poisson Deviance: {poisson_deviance:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE (rounded predictions): {mae:.4f}")

# Get feature importance
importance = best_model.get_booster().get_score(importance_type='gain')
importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
print("\nFeature Importance:")
for feature, score in importance[:10]:
    print(f"{feature}: {score:.2f}")

Loaded saved parameters:
    subsample: 0.6
    reg_lambda: 5
    reg_alpha: 0.5
    n_estimators: 200
    min_child_weight: 1
    max_depth: 3
    learning_rate: 0.1
    gamma: 0.2
    colsample_bytree: 0.6

Final Model Performance:
Poisson Deviance: 1.6003
RMSE: 4.4607
MAE (rounded predictions): 3.4869

Feature Importance:
shots_interaction: 35.74
touches_att_pen_area_interaction: 35.26
is_home: 30.13
touches_att_3rd_interaction: 25.23
shots_on_target_interaction: 22.49
xg_interaction: 21.48
touches_interaction: 14.65
goals_interaction: 14.00
opp_weighted_defense_opp_shots: 12.97
shots_on_target_norm_gap: 12.32


## Predictions

In [6]:
# Generate raw predictions on training data
raw_pred_train = best_model.predict(X_train)

# Generate raw predictions on test data
raw_pred_test = best_model.predict(X_test)

test_indices = np.where(df['match_date'] >= '2025-01-01')[0]

# Create a DataFrame with predictions and relevant information
predictions_df = pd.DataFrame({
    'actual_shots': y_test,
    'predicted_shots': raw_pred_test,
    'prediction_error': raw_pred_test - y_test
})

# Join with the original dataframe to get team information
# Using the test indices to select the right rows from the original df
result_df = pd.concat([
    df.iloc[test_indices].reset_index(drop=True),  # Original data for test set
    predictions_df  # Predictions
], axis=1)


backtest_df = result_df[["match_url", "match_date", "season", "division", "team", "opp_team", "is_home", "predicted_shots", "actual_shots", "prediction_error"]]

backtest_df.to_csv("team_shots_backtest_predictions.csv", index=False)